# OptiMystic JupyterLab Integration Test Notebook

This notebook is for testing and debugging, not for Voila web deployment.

It lets you run and debug each layer independently in JupyterLab:
- Python-only (`cp`)
- Julia-only (`mip`, called through the Python CLI)
- R bridge connectivity (`rpy2` + `r_solvers`)
- Full pipeline (Python/Julia -> R post-processing)

In [19]:
# Step 1: Environment bootstrap
import json
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "examples").exists() and (PROJECT_ROOT.parent / "examples").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "examples"))
import jupyter_debug_tools as jdt
importlib.reload(jdt)

run_python_only_cp_scheduling = jdt.run_python_only_cp_scheduling
run_julia_only_mip_packing = jdt.run_julia_only_mip_packing
run_full_pipeline = jdt.run_full_pipeline
ensure_r_bridge = jdt.ensure_r_bridge
run_r_postprocess = jdt.run_r_postprocess
explain_debug_pipeline = jdt.explain_debug_pipeline
run_section = jdt.run_section
warmup_julia_path = jdt.warmup_julia_path

print("Project root:", PROJECT_ROOT)
print("Loaded helper module from:", jdt.__file__)
print(json.dumps(explain_debug_pipeline(), indent=2))

Project root: C:\Users\kevin\OneDrive\Desktop\OptiMystic
Loaded helper module from: C:\Users\kevin\OneDrive\Desktop\OptiMystic\examples\jupyter_debug_tools.py
{
  "python_only": "Validate only Python CP scheduling logic (excluding Julia/R)",
  "julia_only": "Validate only Julia MIP execution path (called via Python CLI subprocess)",
  "r_bridge": "Validate rpy2 and r_solvers loading/connectivity",
  "full_pipeline": "End-to-end validation: Python/Julia outputs passed into R process_results",
  "quick_mode": "Use quick=True to run smaller payloads for faster feedback",
  "warmup": "Run warmup_julia_path() once to reduce first-run Julia latency"
}


## What Each Section Tests

- Python-only: validates only Python CP logic
- Julia-only: validates only Julia MIP execution path
- R bridge: validates only `rpy2` and `r_solvers` connectivity
- Full pipeline: validates end-to-end handoff from Python/Julia outputs to R post-processing

This structure helps isolate failures quickly by layer.

In [12]:
# Step 2: Fast Python/Julia checks
print("Warming up Julia path (first run can be slower)...")
warmup = warmup_julia_path()
print(json.dumps(warmup, indent=2))

# Quick targeted run
SECTION = "full"  # "python" | "julia" | "full"
section_result = run_section(SECTION, quick=True)
print(f"Section={SECTION}")
print(json.dumps({k: v.get('status') for k, v in section_result.items()}, indent=2))

Warming up Julia path (first run can be slower)...
{
  "status": "Optimal",
  "objective": 10.0,
  "note": "Julia path warmed up. Next runs are usually faster."
}
Section=full
{
  "python_cp": "Optimal",
  "julia_mip": "Optimal"
}


In [20]:
# Step 3: R bridge check
r_info = ensure_r_bridge()
print(json.dumps(r_info, indent=2))

R callback write-console: Learn more about the underlying theory at https://ggplot2-book.org/
  
R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb4 in position 1: invalid start byte <traceback object at 0x0000022F8554D300>
R callback write-console: The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

  


{
  "ok": true,
  "r_version": "R version 4.5.3 (2026-03-11 ucrt)",
  "r_workdir": "C:/Users/kevin/OneDrive/Desktop/OptiMystic/r_solvers"
}


In [22]:
# Step 4: Full pipeline + assertions
pipeline = run_full_pipeline(quick=True)
packing_store = {
    "parameters": {
        "Items": ["A", "B", "C"],
        "Weights": [2, 3, 4],
        "Values": [10, 12, 14],
        "Capacity": 7,
    }
}
packing_processed = run_r_postprocess("packing", pipeline["julia_mip"], packing_store)

print(json.dumps({
    "python_cp_status": pipeline["python_cp"].get("status"),
    "julia_mip_status": pipeline["julia_mip"].get("status"),
    "r_processed_status": packing_processed.get("status"),
}, indent=2))

assert pipeline["python_cp"].get("status") not in ("Error", "error")
assert pipeline["julia_mip"].get("status") not in ("Error", "error")
assert packing_processed.get("status") == "ok"
print("All targeted tests passed (Python/Julia/R).")

{
  "python_cp_status": "Optimal",
  "julia_mip_status": "Optimal",
  "r_processed_status": "ok"
}
All targeted tests passed (Python/Julia/R).


In [25]:
# Step 5: Show R post-analysis details (standardized + visualization)
from pprint import pprint

normalized = {
    "mode": packing_processed.get("mode"),
    "status": packing_processed.get("status"),
    "objective": packing_processed.get("objective", packing_processed.get("total_value")),
    "kpis": {
        "total_value": packing_processed.get("total_value"),
        "used_capacity": packing_processed.get("used_capacity"),
        "capacity": packing_processed.get("capacity"),
        "utilization_pct": (
            round(100.0 * packing_processed.get("used_capacity", 0) / max(packing_processed.get("capacity", 1), 1), 1)
            if packing_processed.get("capacity") is not None
            else None
        ),
    },
    "items": packing_processed.get("items", []),
    "report": packing_processed.get("report"),
}

print("[Standardized R post-analysis]")
pprint(normalized)

items = normalized["items"] or []
if items:
    try:
        import matplotlib.pyplot as plt

        labels = [str(x.get("item", "?")) for x in items]
        values = [float(x.get("value", 0) or 0) for x in items]
        counts = [float(x.get("count", 0) or 0) for x in items]

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

        axes[0].bar(labels, values)
        axes[0].set_title("Item Value")
        axes[0].set_xlabel("Item")
        axes[0].set_ylabel("Value")

        axes[1].bar(labels, counts)
        axes[1].set_title("Item Count")
        axes[1].set_xlabel("Item")
        axes[1].set_ylabel("Count")

        fig.suptitle("Packing Post-Analysis")
        fig.tight_layout()
        plt.show()
    except Exception as exc:
        print("Visualization skipped:", exc)
else:
    print("No selected items returned from post-analysis.")

[Standardized R post-analysis]
{'items': [{'count': 1, 'item': 'A', 'value': 10, 'weight': 2}],
 'kpis': {'capacity': 7,
          'total_value': 10,
          'used_capacity': 2,
          'utilization_pct': 28.6},
 'mode': 'packing',
 'objective': 10,
 'report': 'Total Value: 10.00\nUsed Capacity: 2.00 / 7.00 (28.6%)',
 'status': 'ok'}
Visualization skipped: No module named 'matplotlib'
